# 01 · Build the OULAD master table

**DSP391m · Group 1 · FPT University** — Report 2, Chapter 3 (data collection, integration, cleaning).

This notebook is **self-contained**: every function it uses is defined in the cells below — nothing is imported from `src/`. It integrates the seven raw OULAD tables into one analysis-ready row per *student-module-presentation*, attaches the fixed `at_risk` label (Step-0, Option A: `at_risk=1` if `final_result ∈ {Fail, Withdrawn}`), engineers three feature groups, cleans, and writes `data/interim/master_raw.parquet`.

In [ ]:
import os, json, logging, warnings
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger('nb')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
INTERIM_DIR = ROOT / 'data' / 'interim'
CHECKPOINTS_DIR = ROOT / 'data' / 'checkpoints'
CHECKPOINT_MAP_PATH = ROOT / 'data' / 'checkpoint_map.csv'
REPORTS_DIR = ROOT / 'reports'
TABLES_DIR = REPORTS_DIR / 'tables'
FIGURES_DIR = REPORTS_DIR / 'figures'
CHECKPOINTS = (10, 20, 40, 60, 80, 100)
RANDOM_SEED = 42
for _d in (INTERIM_DIR, CHECKPOINTS_DIR, TABLES_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

## Setup — shared helpers (label, atomic write, raw loader)

`add_at_risk_label` fixes the target; `save_parquet_atomic` writes via a temp file + rename so an interrupted run never corrupts the output; `load_raw_tables` reads the six small CSVs (the 10.6M-row `studentVle` is streamed separately).

In [ ]:
GROUP_COLS = ["code_module", "code_presentation", "id_student"]


PRESENTATION_KEY = ["code_module", "code_presentation"]


AT_RISK_RESULTS = ("Fail", "Withdrawn")


CANONICAL_ACTIVITY_TYPES = (
    "forumng",
    "oucontent",
    "resource",
    "homepage",
    "oucollaborate",
    "quiz",
    "subpage",
    "url",
)


def add_at_risk_label(student_info: pd.DataFrame) -> pd.DataFrame:
    """Append the binary ``at_risk`` column derived from ``final_result``."""
    out = student_info.copy()
    out["at_risk"] = out["final_result"].isin(AT_RISK_RESULTS).astype("int8")
    return out


def save_parquet_atomic(df: pd.DataFrame, path: Path) -> Path:
    """Write a DataFrame to parquet via a temp file + atomic rename.

    A crash mid-write therefore never leaves a half-written, unreadable parquet
    in place of a good one (global checkpointing rule).
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)
    return path


def load_raw_tables(raw_dir: Path = RAW_DIR) -> dict[str, pd.DataFrame]:
    """Load the small OULAD tables (everything except the 432 MB studentVle).

    studentVle is read separately by the engagement builder, which streams it in
    chunks to bound peak memory.
    """
    names = [
        "studentInfo",
        "studentRegistration",
        "studentAssessment",
        "assessments",
        "courses",
        "vle",
    ]
    return {name: pd.read_csv(raw_dir / f"{name}.csv") for name in names}

## Engagement features — aggregate the VLE clickstream

`studentVle` (~10.6M rows) is collapsed to one row per student: `total_clicks`, `n_days_active`, per-activity click counts, `max_clicks_single_day`, `mean_clicks_per_active_day`, and `last_active_day` (helper for `days_since_last_activity`). The aggregation is pure so the checkpoint notebook (03) reuses it on time-sliced clickstreams.

In [ ]:
_STUDENT_VLE_DTYPES = {
    "code_module": "string",
    "code_presentation": "string",
    "id_student": "int32",
    "id_site": "int32",
    "date": "int32",
    "sum_click": "int32",
}


def load_student_vle(raw_dir: Path = RAW_DIR, chunksize: int = 500_000) -> pd.DataFrame:
    """Read studentVle.csv in chunks with simple dtypes.

    Categorical conversion is done *after* the full frame is in memory; building a
    category hash table during the C parse can spike memory on large files.
    """
    path = raw_dir / "studentVle.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    log.info("Reading %s (chunksize=%d)", path.name, chunksize)
    frames = [
        c for c in pd.read_csv(path, dtype=_STUDENT_VLE_DTYPES, chunksize=chunksize)
    ]
    df = pd.concat(frames, ignore_index=True)
    log.info(
        "studentVle: %s rows, %.0f MB",
        f"{len(df):,}",
        df.memory_usage(deep=True).sum() / 1e6,
    )
    return df


def attach_activity_type(
    student_vle: pd.DataFrame, vle_meta: pd.DataFrame
) -> pd.DataFrame:
    """Map id_site -> activity_type via a lookup Series (avoids a 10M-row merge).

    id_site is globally unique in vle.csv, so the lookup is unambiguous.
    """
    if vle_meta["id_site"].duplicated().any():
        raise ValueError("id_site is not unique in vle.csv; lookup-by-site is unsafe")
    site_to_activity = vle_meta.set_index("id_site")["activity_type"]
    out = student_vle.copy()
    out["activity_type"] = out["id_site"].map(site_to_activity)
    n_missing = int(out["activity_type"].isna().sum())
    if n_missing:
        log.warning(
            "%s clicks had no activity_type (id_site absent in vle.csv)",
            f"{n_missing:,}",
        )
    return out


def aggregate_engagement(clickstream: pd.DataFrame) -> pd.DataFrame:
    """Aggregate a (full or cut) clickstream into per-student engagement features.

    ``clickstream`` must contain GROUP_COLS plus ``date``, ``sum_click`` and
    ``activity_type``. Returns one row per student-module-presentation.
    """
    required = set(GROUP_COLS) | {"date", "sum_click", "activity_type"}
    missing = required - set(clickstream.columns)
    if missing:
        raise KeyError(f"clickstream missing columns: {sorted(missing)}")

    grouped = clickstream.groupby(GROUP_COLS, observed=True)
    base = grouped.agg(
        total_clicks=("sum_click", "sum"),
        n_days_active=("date", "nunique"),
        last_active_day=("date", "max"),
    )

    # Clicks per day, then the busiest single day per student.
    daily = clickstream.groupby(GROUP_COLS + ["date"], observed=True)["sum_click"].sum()
    base["max_clicks_single_day"] = daily.groupby(level=GROUP_COLS, observed=True).max()

    # Per-type click counts, restricted to the eight canonical activity types.
    by_type = (
        clickstream.groupby(GROUP_COLS + ["activity_type"], observed=True)["sum_click"]
        .sum()
        .unstack(fill_value=0)
    )
    by_type = by_type.reindex(columns=list(CANONICAL_ACTIVITY_TYPES), fill_value=0)
    by_type.columns = [f"clicks_{c}" for c in by_type.columns]
    base = base.join(by_type)

    base["mean_clicks_per_active_day"] = (
        base["total_clicks"] / base["n_days_active"].where(base["n_days_active"] > 0)
    ).fillna(0.0)

    return base.reset_index()

## Performance features — aggregate assessment submissions

Per student, as of the module-length cutoff (t=100%): `n_assessments_submitted`, `mean_score_to_date`, `weighted_score_to_date`, and `not_submitted` (missed a passed deadline — a known risk signal).

In [ ]:
def aggregate_performance(
    submissions: pd.DataFrame,
    assessments: pd.DataFrame,
    cutoff_lookup: pd.DataFrame,
    roster: pd.DataFrame,
) -> pd.DataFrame:
    """Build per-student performance features as of each presentation's cutoff.

    Parameters
    ----------
    submissions   : studentAssessment (id_assessment, id_student, date_submitted,
                    is_banked, score).
    assessments   : assessments.csv (id_assessment, code_module, code_presentation,
                    assessment_type, date [deadline, NaN for the final exam], weight).
    cutoff_lookup : PRESENTATION_KEY + ``cutoff_day`` (module length for t=100%,
                    or the checkpoint day).
    roster        : every student to emit a row for (GROUP_COLS); guarantees
                    non-submitters still receive features (and not_submitted).
    """
    meta = assessments.merge(cutoff_lookup, on=PRESENTATION_KEY, how="left")
    # An assessment is "due to date" when it has a real deadline on/before cutoff.
    meta["is_due"] = meta["date"].notna() & (meta["date"] <= meta["cutoff_day"])
    due_per_pres = (
        meta[meta["is_due"]].groupby(PRESENTATION_KEY).size().rename("n_due_to_date")
    )

    sub = submissions.merge(
        meta[["id_assessment", *PRESENTATION_KEY, "weight", "cutoff_day", "is_due"]],
        on="id_assessment",
        how="left",
    )
    # Submissions counted "to date": real (not banked) and submitted by cutoff.
    submitted = sub[
        (sub["is_banked"] == 0) & (sub["date_submitted"] <= sub["cutoff_day"])
    ].copy()
    submitted["weighted"] = submitted["score"] * submitted["weight"] / 100.0

    agg = submitted.groupby(GROUP_COLS).agg(
        n_assessments_submitted=("id_assessment", "count"),
        mean_score_to_date=("score", "mean"),
        weighted_score_to_date=("weighted", "sum"),
    )
    # Of those, how many were for assessments whose deadline had passed.
    submitted_due = (
        submitted[submitted["is_due"]]
        .groupby(GROUP_COLS)
        .size()
        .rename("n_submitted_due")
    )

    out = roster[GROUP_COLS].drop_duplicates().copy()
    out = out.merge(agg, on=GROUP_COLS, how="left")
    out = out.merge(submitted_due, on=GROUP_COLS, how="left")
    out = out.merge(due_per_pres, on=PRESENTATION_KEY, how="left")

    out["n_assessments_submitted"] = (
        out["n_assessments_submitted"].fillna(0).astype("int32")
    )
    out["mean_score_to_date"] = out["mean_score_to_date"].fillna(0.0)
    out["weighted_score_to_date"] = out["weighted_score_to_date"].fillna(0.0)
    n_due = out["n_due_to_date"].fillna(0)
    n_submitted_due = out["n_submitted_due"].fillna(0)
    out["not_submitted"] = ((n_due - n_submitted_due) > 0).astype("int8")

    return out.drop(columns=["n_submitted_due", "n_due_to_date"])

## Cleaning helpers (Task 5)

Drop duplicate composite keys and standardise categorical text; `imd_band` codes such as `10-20` are kept verbatim because the ordinal encoder expects them.

In [ ]:
CATEGORICAL_COLS = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]


def _clean(master: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Task 5: drop duplicate keys and standardise categorical text."""
    rows = []
    n_dup = _dup_count(master)
    master = master.drop_duplicates(subset=GROUP_COLS).reset_index(drop=True)
    rows.append({"item": "duplicate_keys_removed", "value": n_dup})

    for col in CATEGORICAL_COLS:
        if col in master.columns and master[col].dtype == object:
            master[col] = master[col].str.strip()
            rows.append(
                {"item": f"{col}_n_unique", "value": master[col].nunique(dropna=True)}
            )

    return master, pd.DataFrame(rows)


def _dup_count(df: pd.DataFrame) -> int:
    return int(df.duplicated(subset=GROUP_COLS).sum())


def _feature_nans(df: pd.DataFrame) -> int:
    cols = [c for c in df.columns if c not in ("imd_band", "date_unregistration")]
    return int(df[cols].isnull().sum().sum())

## Run the pipeline

**Step 1 — load raw tables and fix the label.**

In [ ]:
raw = load_raw_tables(RAW_DIR)
student_info = add_at_risk_label(raw['studentInfo'])
courses = raw['courses']
print('studentInfo:', student_info.shape, '| at_risk rate: '
      f"{student_info['at_risk'].mean():.1%}")

**Step 2 — engagement.** Read the clickstream (parquet cache if present, else the CSV in chunks), map `id_site → activity_type`, and aggregate.

In [ ]:
vle_parquet = INTERIM_DIR / 'studentVle.parquet'
if vle_parquet.exists():
    student_vle = pd.read_parquet(vle_parquet)
    print('loaded cached clickstream parquet:', student_vle.shape)
else:
    student_vle = load_student_vle(RAW_DIR)
vle_meta = pd.read_csv(RAW_DIR / 'vle.csv')
clickstream = attach_activity_type(student_vle, vle_meta)
engagement = aggregate_engagement(clickstream)
print('engagement:', engagement.shape)
engagement.head(3)

**Step 3 — performance.** Cutoff = full module length (t=100%).

In [ ]:
cutoff_lookup = courses[PRESENTATION_KEY + ['module_presentation_length']].rename(
    columns={'module_presentation_length': 'cutoff_day'})
performance = aggregate_performance(
    raw['studentAssessment'], raw['assessments'], cutoff_lookup, roster=student_info)
print('performance:', performance.shape)
performance.head(3)

**Step 4 — integrate by audited left-joins.** Each join is validated `many_to_one`; a before/after row-count log proves no duplication or loss (the population stays 32,593).

In [ ]:
registration = raw['studentRegistration']
join_log = []

def _merge(left, right, cols, name):
    before = len(left)
    merged = left.merge(right, on=cols, how='left', validate='many_to_one')
    join_log.append({'step': name, 'rows_before': before, 'rows_after': len(merged),
                     'n_students': merged['id_student'].nunique(),
                     'delta': len(merged) - before})
    return merged

master = student_info.copy()
join_log.append({'step': 'studentInfo(base)', 'rows_before': len(master),
                 'rows_after': len(master), 'n_students': master['id_student'].nunique(),
                 'delta': 0})
master = _merge(master, registration[PRESENTATION_KEY + ['id_student',
                'date_registration', 'date_unregistration']], GROUP_COLS, 'registration')
master = _merge(master, engagement, GROUP_COLS, 'engagement')
master = _merge(master, performance, GROUP_COLS, 'performance')
master = _merge(master, courses[PRESENTATION_KEY + ['module_presentation_length']],
                PRESENTATION_KEY, 'courses')
display(pd.DataFrame(join_log))

**Step 5 — derive `days_since_last_activity`, fill engagement gaps, clean.**

A student absent from `studentVle` simply had no activity → engagement counts fill to 0; but their inactivity gap is the whole window, so `days_since_last_activity` is derived from `last_active_day` (not filled with 0).

In [ ]:
master['days_since_last_activity'] = (
    master['module_presentation_length'] - master['last_active_day']).clip(lower=0)
master['days_since_last_activity'] = master['days_since_last_activity'].fillna(
    master['module_presentation_length'])
master = master.drop(columns=['last_active_day'])

engagement_fill = [c for c in engagement.columns
                   if c not in GROUP_COLS and c != 'last_active_day']
master[engagement_fill] = master[engagement_fill].fillna(0)

master, cleaning_log = _clean(master)
display(cleaning_log)

**Step 6 — validate and persist.**

In [ ]:
save_parquet_atomic(master, INTERIM_DIR / 'master_raw.parquet')
pd.DataFrame(join_log).to_csv(INTERIM_DIR / 'master_join_log.csv', index=False)
cleaning_log.to_csv(INTERIM_DIR / 'master_cleaning_log.csv', index=False)

key = GROUP_COLS
assert len(master) == 32593 and master.duplicated(key).sum() == 0
print(f'master_raw: {len(master):,} rows x {master.shape[1]} cols')
print(f"at_risk rate: {master['at_risk'].mean():.1%} | duplicate keys: "
      f'{master.duplicated(key).sum()} | feature NaNs: {_feature_nans(master)}')
print('OK — 32,593 unique records, no duplicate keys')
master.head(3)

## Conclusion

`master_raw` (32,593 × 33) integrates demographic, engagement and performance features with the fixed label and audited join integrity. It is the single input to the EDA (notebook 02), the time-aware checkpoints (notebook 03), and preprocessing (notebook 04).